# FunnelIQ — Design Decisions Explained

This notebook is a **study companion**, not project code. It walks through every architecture,
data, and modeling decision made so far for the FunnelIQ final project, explains *why* each
decision was made, and re-runs the actual verification checks against the real dataset so you
can reproduce the numbers yourself.

Use it to refresh your own understanding before explaining the project to your instructor —
every claim below is either a design decision (with its reasoning) or a number you can
recompute by running the code cells.

**Companion files in this repo:**
- `FunnelIQ_Assignment.html` — the original project brief
- `funneliq_leakage_decisions.md` — the leakage section for `REPORT.md`
- `schema.sql` — the Supabase table + RLS policies
- `funnel_marketing_data.csv` — the raw dataset (3,500 rows)


## 1. The scenario, in one paragraph

Northbound Media (a 40-person marketing agency) has two years of funnel data in a spreadsheet
nobody trusts. The founder wants a **deployed, login-gated tool** — not a notebook — that can
answer: how long will a customer stay, who will upsell, who becomes a "super customer" (stays +
spends + refers), where should ad budget go, and are late follow-ups actually wasted effort.

That last requirement (a real deployed tool, not a slide deck) is why the project has three
infrastructure "pillars" alongside the six modeling work packages.


## 2. Architecture decisions

**Two backends, one frontend:**
- **Supabase** — Postgres for the dataset, plus Supabase Auth for the login screen. Row Level
  Security (RLS) is enabled so the *database itself* enforces that only signed-in users can read
  data — not just the app code.
- **Railway API** — prediction only. It loads pre-trained models and serves predictions; it does
  not own auth or raw data storage.

**Why split it this way:** the brief's core auth requirement is that RLS actually has teeth —
"a backend that reads everything with the service key bypasses your policies." Keeping the
service key **local** (used only for the one-time data-load script) and never shipping it to
Railway or the browser is what makes that requirement real rather than theoretical.

**Offline / runtime split:** models are trained **offline**, ahead of time, and saved as `.pkl`
files. The deployed Railway server only **loads and serves** them — it never trains at request
time. This keeps the API stateless, fast to start, and safe to restart (matches the brief's
"deploy an empty skeleton on day one, health-check first" advice).

**Open item:** the exact mechanism for the Railway API to respect a user's session (does it
re-verify the Supabase JWT itself, or rely on the frontend only calling it when signed in?) isn't
fully pinned down yet — worth deciding explicitly before deployment.


In [1]:
# Setup — load the real dataset so every number below is reproducible.
import pandas as pd

df = pd.read_csv("funnel_marketing_data.csv")
print(f"rows: {len(df)}, columns: {len(df.columns)}")
df.head(3)


rows: 3500, columns: 19


,ad_budget,num_leads,leads_answered,leads_not_answered,followup_1,followup_2,followup_3,followup_4,followup_5,not_closed,closed,calls_to_closed,calls_to_not_closed,customer_acquisition_cost,ltv_months,purchased,upsell,cumulative_profit,referred
0,2500,36,24,12,19,14,11,10,7,5,2,2,4,1250,38.0,1,0,20777.0,No
1,15000,98,55,43,43,32,26,25,18,14,4,5,3,3750,10.0,1,0,3531.0,Yes
2,6000,60,35,25,27,19,16,15,11,7,4,5,3,1500,12.0,1,0,3967.0,No


## 3. Data quality findings

Confirmed once during design, re-verified below. These numbers feed directly into the schema
(`schema.sql`) and the leakage decisions (`funneliq_leakage_decisions.md`).


In [2]:
# Nulls per column — only two columns should have any.
nulls = df.isna().sum()
nulls[nulls > 0]


ltv_months            4
cumulative_profit    29
dtype: int64

**Decision this drives:** in `schema.sql`, every column is `not null` **except** `ltv_months`
and `cumulative_profit`, which allow nulls. If this cell ever shows a different set of columns
with nulls, the schema needs to change too.


In [3]:
# Exact duplicate rows.
dup_count = df.duplicated().sum()
print(f"duplicate rows: {dup_count}")


duplicate rows: 10


**Status: still an open decision.** 10 duplicate rows exist in the source CSV, but neither
`schema.sql` nor the (not-yet-written) load script currently dedupes them. Most likely fix:
`DISTINCT` / `drop_duplicates()` in the load script before inserting into Supabase.


In [4]:
# num_leads: is it really just "collinear" with the other lead columns, or something stronger?
mismatch = (df["num_leads"] - (df["leads_answered"] + df["leads_not_answered"])) != 0
print(f"rows where num_leads != leads_answered + leads_not_answered: {mismatch.sum()}")


rows where num_leads != leads_answered + leads_not_answered: 0


**Finding:** this isn't approximate correlation — it's an **exact linear identity** for all
3,500 rows. `num_leads` is fully determined by the other two columns.

**Why it matters:** keeping all three as model features anywhere is redundant (perfect
multicollinearity). Harmless for the tree-based models this project uses (XGBoost / LightGBM /
CatBoost split on whichever of the three is convenient), but worth a one-line note in the
`REPORT.md` feature-importance discussion — a low importance score on one of the three doesn't
mean it "doesn't matter," it means its information is being captured through a sibling column.


In [5]:
# column dtypes actually contain what the schema assumes (no hidden decimals in "integer" columns)
candidates = ["calls_to_closed", "calls_to_not_closed", "customer_acquisition_cost", "ad_budget"]
for col in candidates:
    non_integer = (df[col].dropna() % 1 != 0).sum()
    print(f"{col}: non-integer values = {non_integer}")


calls_to_closed: non-integer values = 0
calls_to_not_closed: non-integer values = 0
customer_acquisition_cost: non-integer values = 0
ad_budget: non-integer values = 0


**Why this check exists:** the brief describes `calls_to_closed` / `calls_to_not_closed` as
*"average calls"* — averages are usually decimals, which made `schema.sql`'s `integer` typing
look suspicious at first. Turns out this dataset's averages happen to be whole numbers, so
`integer` is correct as written. **This was an initial concern that the data verification
disproved — a good example of why every schema assumption should be checked against the real
file, not just against how a column is described in English.**


## 4. The leakage framework

Every modeling package (2, 3, 4, 6) is checked against one test:

> **"Is this value already known at the moment I make the prediction?"**
> If no — it only fills in *later* — it's leakage. Exclude it.

Plus a rule that catches most of this dataset's traps:

> **An outcome cannot predict another outcome.**
> `ltv_months`, `upsell`, `cumulative_profit`, and `referred` are all "how it ended." Using one
> to predict another is leakage, even when the correlation looks great.

### Column tiers (when does each value become known?)

| Tier | Meaning | Columns |
|---|---|---|
| 1. Funnel inputs | known before/at campaign start | `ad_budget`, `num_leads`, `leads_answered`, `leads_not_answered` |
| 2. Funnel process | known during the sales process | `followup_1`…`followup_5`, `calls_to_closed`, `calls_to_not_closed`, `closed`, `not_closed` |
| 3. Acquisition facts | known at conversion | `customer_acquisition_cost`, `purchased` |
| 4. Lifetime outcomes | known only after the relationship plays out | `upsell`, `ltv_months`, `cumulative_profit`, `referred` |

**Rule of thumb:** Tier 4 columns are outcomes. In any prediction package, at most one of them
is the target — the rest are off-limits.


## 5. Package 2 — LTV regression (target: `ltv_months`)

**Assumed prediction moment:** at/just after acquisition — "we just got this customer, how long
will they stay?"

- **Allowed:** Tier 1, 2, 3.
- **Excluded:** `upsell`, `cumulative_profit`, `referred` (sibling lifetime outcomes).

**The brief directly asks: "Should `cumulative_profit` be a feature here?" → No.**


In [6]:
# Why cumulative_profit is excluded from the LTV model: check the correlation.
corr = df[["cumulative_profit", "ltv_months"]].corr().iloc[0, 1]
print(f"correlation(cumulative_profit, ltv_months) = {corr:.4f}")


correlation(cumulative_profit, ltv_months) = 0.8459


Profit accrues over the customer's *whole* lifetime, so it isn't known at the moment you're
predicting lifetime — and the 0.85 correlation above shows it's also mechanically tied to
tenure (longer stay → more profit). Including it wouldn't teach the model anything real; it
would just let the model "cheat" by looking at a near-restatement of the answer. Textbook
leakage, confirmed by the correlation.


## 6. Package 3 — Upsell classification (target: `upsell`)

**Assumed prediction moment:** after purchase, to decide who to target with an upsell offer.

- **Allowed:** Tier 1, 2, 3 — **computed only on the `purchased == 1` subset.**
- **Excluded:** `ltv_months`, `cumulative_profit`, `referred`.
- **`purchased` itself:** used to *filter* the rows, then **dropped** as a feature (see below).

This is the package where the design decision changed after checking the real data — worth
understanding in detail since it's a good story for your instructor about why you verify
assumptions against data instead of trusting a column description.


In [7]:
# The purchased -> upsell relationship, in full.
pd.crosstab(df["purchased"], df["upsell"], margins=True)


upsell,0,1,All
purchased,,,
0,337,0,337
1,1697,1466,3163
All,2034,1466,3500


**Reading this table:** every single one of the 337 `purchased == 0` rows has `upsell == 0` —
zero exceptions. That makes sense logically: you can't buy "additional" services without an
initial purchase, so `upsell` is *deterministically* 0 whenever `purchased == 0`.

**Why that's a problem if left unfiltered:** if you train a classifier on the full dataset,
`purchased` becomes a trivial near-perfect predictor of `upsell == 0` for those 337 rows. Every
evaluation metric (accuracy, F1, ROC-AUC) would look inflated — but the model would have learned
almost nothing about what actually drives upselling among people who *did* buy, which is the
business question that actually matters.

**The decision:** filter the training set to `purchased == 1` **before** training, then drop
`purchased` as a feature (it's now constant within the filtered set, so it carries no signal
anyway). Below, confirm the filtered target is still a meaningful, well-balanced problem —
not something that got trivial by filtering.


In [8]:
# Class balance within the purchased == 1 subset — is the filtered problem still meaningful?
purchasers = df[df["purchased"] == 1]
print(f"purchased == 1 rows: {len(purchasers)}")
print(purchasers["upsell"].value_counts())


purchased == 1 rows: 3163
upsell
0    1697
1    1466
Name: count, dtype: int64


1,697 vs. 1,466 — a healthy, well-balanced split. The filtered task ("who among people who
already bought will also upsell?") is the meaningful modeling problem; the unfiltered task
("did they buy at all, disguised as an upsell question") is not.


## 7. Package 4 — The "super customer" score (target: `referred`)

**Prediction moment, from the brief itself:** *"given a new customer's early funnel data, output
a 0–100 likelihood."* This is the strictest package — only early-funnel data is fair game.

- **Allowed:** Tier 1 (early funnel) + an engineered budget-tier feature (Low / Mid / High from
  `ad_budget` — fine, since it's derived from an allowed input).
- **Excluded:** `ltv_months`, `cumulative_profit`, `upsell`, and of course `referred` itself.

**The trap to say out loud:** the brief *defines* a super customer as someone who
`referred = Yes` **and** `upsell = 1` **and** has long tenure. Two of those three conditions —
`upsell` and `ltv_months` — are not just "outcome columns" in the generic sense, they are
literally **part of the definition of the thing you're trying to predict early.** Using them
wouldn't just leak information, it would make the whole exercise circular: you'd be scoring
customers on traits you only learn *after* they already are or aren't super customers, which
defeats the entire point of "spot them earlier."

**Still conditional (⚠️):** full-funnel signals like `closed`, `calls_*`, and
`customer_acquisition_cost` happen *after* "early" in the funnel — decide per feature how early
"early" really is for your definition, and write down the reasoning.


## 8. Package 6 — Budget optimization (target: `cumulative_profit`)

**This is not classic leakage — it's an availability-at-inference constraint.** The brief says
the simulator *"only knows each campaign's budget."*

- **Excluded regardless (Tier 4 siblings):** `ltv_months`, `upsell`, `referred`.
- **The real constraint:** funnel features (`num_leads`, `closed`, …) are legitimately correlated
  with profit and are fine to **train** on — but at **simulation** time, only `ad_budget` will be
  known. Two valid designs:
  - **(a) Budget-only model** — train on `ad_budget` (and its derived tier) alone. Simplest, no
    imputation needed.
  - **(b) Full-funnel model + imputation** — train on all funnel features, then at simulation
    time fill them in from a "typical funnel profile per budget level" (the brief's own
    suggestion). More realistic, more work.
  - Either is acceptable — the deliverable is picking one and writing down why.


## 9. Packages 1 & 5 — descriptive, not predictive

Package 1 (EDA/cleaning) and Package 5 (the follow-up-dropout "paradox") don't have a held-out
prediction target, so the leakage rule doesn't apply the same way. Package 5 legitimately *uses*
`closed` to study which follow-up stage matters most — that's fine, because you're **describing**
what already happened, not **forecasting** an unknown future outcome. Don't over-apply the
leakage rule to purely descriptive analysis.


## 10. Schema decisions (`schema.sql`)

One table, `funnel_records`, loaded **as-is** from the CSV — no transforms at load time, so the
load script stays simple and repeatable. Cleaning for modeling happens **offline**, per work
package; the table itself is the raw source of truth the app reads at runtime.

Key points:
- Every column is `not null` except `ltv_months` and `cumulative_profit` (matches section 3's
  null counts exactly).
- `purchased` and `upsell` are `smallint not null` (0/1); `referred` is `text not null`
  (`'Yes'`/`'No'`) — matches the brief's column descriptions.
- **Row Level Security** is enabled, with a single `select` policy for `authenticated` users and
  **no** insert/update/delete policy for normal users. The one-time data load runs from a
  developer machine using the **service key**, which bypasses RLS — so it doesn't need its own
  policy, and regular signed-in app users can only ever read, never write.


In [9]:
with open("schema.sql") as f:
    print(f.read())


-- FunnelIQ — database schema
-- Run this in the Supabase SQL editor (or via the CLI) once, before loading data.
-- One table holds the dataset; the app reads from it at runtime for insight panels.
-- Models are trained OFFLINE from the CSV/table — nothing here trains anything.

-- ----------------------------------------------------------------------------
-- 1. The dataset table
-- ----------------------------------------------------------------------------
-- Columns are loaded AS-IS from funnel_marketing_data.csv (no transforms) so the
-- load script stays simple and repeatable. Cleaning for MODELING happens offline,
-- per package — the table is the raw source of truth the app surfaces.

create table if not exists funnel_records (
  id                          bigint generated always as identity primary key,

  -- Tier 1: funnel inputs
  ad_budget                   integer  not null,
  num_leads                   integer  not null,
  leads_answered              integer  not null,


## 11. Summary matrix (cheat sheet)

Legend: ✅ allowed feature · ❌ leakage / exclude · ⚠️ conditional (see package notes) ·
🔒 used to filter rows, then dropped (not a model feature) · **T** = target

| Column | P2 `ltv_months` | P3 `upsell` | P4 `referred` (early) | P6 `cumulative_profit` |
|---|:--:|:--:|:--:|:--:|
| `ad_budget` | ✅ | ✅ | ✅ | ✅ (sim input) |
| `num_leads` | ✅ | ✅ | ✅ | ⚠️ |
| `leads_answered` / `leads_not_answered` | ✅ | ✅ | ✅ | ⚠️ |
| `followup_1…5` | ✅ | ✅ | ⚠️ | ⚠️ |
| `calls_to_closed` / `calls_to_not_closed` | ✅ | ✅ | ⚠️ | ⚠️ |
| `closed` / `not_closed` | ✅ | ✅ | ⚠️ | ⚠️ |
| `customer_acquisition_cost` | ✅ | ✅ | ⚠️ | ⚠️ |
| `purchased` | ✅ | 🔒 filter, not a feature | ⚠️ | ⚠️ |
| `ltv_months` | **T** | ❌ | ❌ | ❌ |
| `upsell` | ❌ | **T** | ❌ | ❌ |
| `cumulative_profit` | ❌ | ❌ | ❌ | **T** |
| `referred` | ❌ | ❌ | **T** | ❌ |


## 12. Review pass — what got corrected, and what got double-checked and confirmed fine

When these decisions were first reviewed, three things were flagged as *possible* issues. Only
one turned out to be real once checked against the actual CSV — a useful reminder that a
"looks suspicious" flag is a hypothesis to test, not a conclusion.

| # | Flagged concern | Verdict after checking the CSV | Outcome |
|---|---|---|---|
| 1 | Package 3's leakage doc didn't mention filtering to `purchased == 1`, even though that was the design decision | **Confirmed real** — `purchased == 0` deterministically implies `upsell == 0` (section 6) | Fixed: `funneliq_leakage_decisions.md` now states the filter explicitly, with the crosstab numbers as evidence |
| 2 | `calls_to_closed` / `calls_to_not_closed` typed `integer`, but described as "averages" (which are usually decimal) | **False alarm** — verified zero non-integer values in the real data | No change — `schema.sql` was already correct |
| 3 | `customer_acquisition_cost` typed `integer`, currency sometimes has decimals | **False alarm** — verified zero non-integer values | No change — `schema.sql` was already correct |

**Still open (not a wrong decision, just undecided):** the 10 duplicate rows found in section 3
have no dedupe step defined yet in either `schema.sql` or the (not-yet-written) load script.


## 13. What's still left to build

This notebook only covers **decisions**, not implementation. Per the brief, still outstanding:

- **GitHub pillar:** repo, `.gitignore`, CI workflow, feature-branch + PR workflow.
- **Supabase pillar:** provision the actual project, run `schema.sql`, write the repeatable
  CSV-load script (with the dedupe decision from section 3/12 baked in), build the login screen.
- **Railway pillar:** deploy an empty "hello + health check" skeleton first, then connect
  GitHub auto-deploy, then add the real prediction endpoints once models are trained offline.
- **Six work packages:** none of the actual modeling has started yet — this notebook is the
  leakage/schema groundwork the packages will build on.
